# Public pancreatic-development workflow: ScGeo representation dynamics

This notebook uses original public pancreas cluster annotations as the principal biological reference. It defines prespecified developmental cluster-transition edges, computes transition vectors between state centers, compares source-state RNA velocity directions to those transition vectors across PCA20, PCA30, PCA50, diffusion map, and UMAP diagnostics, and records negative controls.

No artificial treatment/control labels are created. The classifications use prespecified ScGeo alignment defaults and do not tune thresholds.

## Reader guide

- **Purpose:** Public pancreatic-development workflow: ScGeo representation dynamics in the frozen revision workflow.
- **Inference scope:** Descriptive public developmental-dynamics validation. CellRank provides velocity-derived context, not independent biological confirmation.
- **Inputs:** The official public pancreas input or the checksum-validated output of the preceding numbered stage.
- **Implementation:** `scripts/pancreas_validation_common.py`.
- **Outputs:** Ignored `results/public_validation/pancreas_dataset_d/` artifacts.
- **Frozen findings:** Retain the frozen descriptive representation–dynamics findings and negative controls; do not claim causal trajectories or universal biological preservation.
- **Limitations:** The workflow is descriptive, depends on supplied dynamics and representations, and does not provide independent biological replication.

This source notebook is intentionally a thin, output-free entry point. The testable implementation is maintained in scripts/pancreas_validation_common.py. Executed review copies and generated artifacts are written under the ignored results directory.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from pancreas_validation_common import (
    configured_paths,
    ensure_output_tree,
    ensure_runtime_env,
    load_config,
    rel_display,
    sha256_file,
    version_record,
    write_alt_text,
    write_dataframe,
    write_json,
    write_metadata,
)

CONFIG = load_config(ROOT)
PATHS = configured_paths(CONFIG, ROOT)
DATA_DIR = PATHS["data_dir"]
OUTPUT_DIR = PATHS["output_dir"]
DATA_DIR.mkdir(parents=True, exist_ok=True)
ensure_runtime_env(OUTPUT_DIR)
ensure_output_tree(OUTPUT_DIR)

In [ ]:

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

from pancreas_validation_common import classify_cosine, clean_label, consensus_from_rows, cosine

scvelo_summary = pd.read_csv(OUTPUT_DIR / "figure_sources" / "01_scvelo_summary.csv").iloc[0]
adata = ad.read_h5ad(ROOT / scvelo_summary["output_h5ad"])
cluster_key = CONFIG["cluster_key"]
clusters = adata.obs[cluster_key].astype(str)
min_cells = int(CONFIG["scgeo_alignment_defaults"]["min_cells_per_cluster"])

if "X_pca" not in adata.obsm or adata.obsm["X_pca"].shape[1] < 50:
    sc.tl.pca(adata, n_comps=50, svd_solver="arpack", random_state=CONFIG["scvelo"]["random_state"])
for rep in CONFIG["representations"]:
    if rep["source_obsm_key"] == "X_pca":
        n = int(rep["n_components"])
        adata.obsm[rep["obsm_key"]] = np.asarray(adata.obsm["X_pca"])[:, :n].copy()
        if "velocity_pca" in adata.obsm:
            adata.obsm[rep["velocity_key"]] = np.asarray(adata.obsm["velocity_pca"])[:, :n].copy()

velocity_projection_warnings = []
if "X_diffmap" not in adata.obsm:
    sc.tl.diffmap(adata, n_comps=CONFIG["representations"][3]["n_components"])
try:
    if "velocity_diffmap" not in adata.obsm:
        scv.tl.velocity_embedding(adata, basis="diffmap", all_comps=True)
except Exception as exc:
    velocity_projection_warnings.append(f"velocity_diffmap unavailable: {type(exc).__name__}: {exc}")

rows = []
control_rows = []
rng = np.random.default_rng(CONFIG["negative_controls"]["seed"])

def get_matrix(key, n=None):
    if key not in adata.obsm:
        return None
    arr = np.asarray(adata.obsm[key], dtype=float)
    return arr[:, :n] if n is not None else arr

def orthogonal_rotation(n, seed_offset):
    local = np.random.default_rng(CONFIG["negative_controls"]["seed"] + seed_offset)
    q, r = np.linalg.qr(local.normal(size=(n, n)))
    signs = np.sign(np.diag(r))
    signs[signs == 0] = 1
    return q * signs

def compute_one(edge, rep, X, V, control_type="forward", shuffled_V=None, rotated_Q=None):
    source = edge["source"]
    target = edge["target"]
    if control_type == "reversed_edges":
        source, target = target, source
    source_mask = clusters.eq(source).to_numpy()
    target_mask = clusters.eq(target).to_numpy()
    status = "usable"
    reasons = []
    if source not in set(clusters):
        status = "unavailable"; reasons.append("source cluster missing")
    if target not in set(clusters):
        status = "unavailable"; reasons.append("target cluster missing")
    if source_mask.sum() < min_cells:
        status = "unavailable"; reasons.append("source under min_cells")
    if target_mask.sum() < min_cells:
        status = "unavailable"; reasons.append("target under min_cells")
    if X is None:
        status = "unavailable"; reasons.append("representation missing")
    if V is None:
        status = "unavailable"; reasons.append("velocity representation missing")
    if status == "usable":
        velocity_matrix = V
        if control_type == "shuffled_velocity":
            velocity_matrix = shuffled_V
        elif control_type == "rotated_velocity":
            velocity_matrix = V @ rotated_Q
        source_center = X[source_mask].mean(axis=0)
        target_center = X[target_mask].mean(axis=0)
        transition_vector = target_center - source_center
        velocity_vector = velocity_matrix[source_mask].mean(axis=0)
        transition_norm = float(np.linalg.norm(transition_vector))
        velocity_norm = float(np.linalg.norm(velocity_vector))
        cos_value = cosine(transition_vector, velocity_vector)
        cls = classify_cosine(cos_value, CONFIG)
        if not np.isfinite(cos_value):
            status = "unavailable"; reasons.append("zero or nonfinite transition/velocity norm")
            cls = "unavailable"
    else:
        transition_norm = np.nan
        velocity_norm = np.nan
        cos_value = np.nan
        cls = "unavailable"
    return {
        "transition_id": edge["transition_id"],
        "source": source,
        "target": target,
        "original_source": edge["source"],
        "original_target": edge["target"],
        "biological_rationale": edge["rationale"],
        "representation": rep["name"],
        "principal_representation": bool(rep["principal"]),
        "display_only": bool(rep["display_only"]),
        "control_type": control_type,
        "transition_norm": transition_norm,
        "velocity_norm": velocity_norm,
        "cosine": cos_value,
        "class": cls,
        "status": status,
        "status_reason": "; ".join(reasons) if reasons else "usable",
        "n_source_cells": int(source_mask.sum()),
        "n_target_cells": int(target_mask.sum()),
    }

for rep_i, rep in enumerate(CONFIG["representations"]):
    n = int(rep["n_components"])
    X = get_matrix(rep["obsm_key"], n)
    V = get_matrix(rep["velocity_key"], n)
    shuffled_V = None if V is None else V[rng.permutation(V.shape[0])]
    rotated_Q = None if V is None else orthogonal_rotation(V.shape[1], rep_i)
    for edge in CONFIG["transition_edges"]:
        forward = compute_one(edge, rep, X, V, "forward")
        rows.append(forward)
        for control in CONFIG["negative_controls"]["controls"]:
            if control == "reversed_edges":
                control_rows.append(compute_one(edge, rep, X, V, control))
            elif control == "shuffled_velocity":
                control_rows.append(compute_one(edge, rep, X, V, control, shuffled_V=shuffled_V))
            elif control == "rotated_velocity":
                control_rows.append(compute_one(edge, rep, X, V, control, rotated_Q=rotated_Q))

alignment = pd.DataFrame(rows)
controls = pd.DataFrame(control_rows)
consensus_rows = []
for transition_id, group in alignment.groupby("transition_id", sort=False):
    consensus = consensus_from_rows(group, CONFIG)
    first = group.iloc[0]
    consensus_rows.append({
        "transition_id": transition_id,
        "source": first["original_source"],
        "target": first["original_target"],
        "biological_rationale": first["biological_rationale"],
        **consensus,
    })
agreement = pd.DataFrame(consensus_rows)

cluster_fate_path = OUTPUT_DIR / "figure_sources" / "02_cellrank_cluster_fate_summary.csv"
cluster_fate = pd.read_csv(cluster_fate_path) if cluster_fate_path.exists() else pd.DataFrame()
cellrank_rows = []
for edge in CONFIG["transition_edges"]:
    source = edge["source"]
    target = edge["target"]
    status = "unavailable"
    score = np.nan
    reason = "CellRank fate summary unavailable"
    if not cluster_fate.empty and cluster_key in cluster_fate.columns and source in set(cluster_fate[cluster_key].astype(str)):
        source_row = cluster_fate.loc[cluster_fate[cluster_key].astype(str).eq(source)].iloc[0]
        matching_cols = [col for col in cluster_fate.columns if col != cluster_key and target.lower() in col.lower()]
        if matching_cols:
            score = float(source_row[matching_cols].max())
            status = "available_velocity_derived_comparator"
            reason = f"matched CellRank fate column(s): {', '.join(matching_cols)}"
        else:
            status = "not_direct_terminal_fate_or_unmatched_name"
            reason = "target does not directly match a CellRank terminal fate column"
    cellrank_rows.append({
        "transition_id": edge["transition_id"],
        "source": source,
        "target": target,
        "cellrank_comparator_status": status,
        "cellrank_source_to_target_fate_score": score,
        "cellrank_status_reason": reason,
        "cellrank_independent_of_scvelo": False,
    })
cellrank_compare = pd.DataFrame(cellrank_rows)

evidence = agreement.merge(cellrank_compare, on=["transition_id", "source", "target"], how="left")
control_summary = controls.groupby(["transition_id", "control_type"], observed=False).agg(
    n_usable=("status", lambda x: int((x == "usable").sum())),
    median_cosine=("cosine", "median"),
    aligned_fraction=("class", lambda x: float((x == "aligned").mean())),
).reset_index()
forward_summary = alignment[alignment["principal_representation"]].groupby("transition_id", observed=False).agg(
    forward_median_cosine=("cosine", "median"),
    forward_aligned_fraction=("class", lambda x: float((x == "aligned").mean())),
).reset_index()
evidence = evidence.merge(forward_summary, on="transition_id", how="left")
negative_aligned = control_summary.groupby("transition_id")["aligned_fraction"].mean().to_dict()
evidence["negative_control_aligned_fraction"] = evidence["transition_id"].map(negative_aligned)
evidence["negative_control_specificity"] = 1.0 - evidence["negative_control_aligned_fraction"]

def limitation_reason(row):
    reasons = []
    if row["consensus_class"] in ["unstable", "unavailable"]:
        reasons.append(str(row["status_reason"]))
    if pd.notna(row.get("negative_control_aligned_fraction")) and float(row["negative_control_aligned_fraction"]) > 0:
        reasons.append(f"negative controls retain aligned calls in {float(row['negative_control_aligned_fraction']):.2f} of tested representation-control cases")
    if row.get("cellrank_comparator_status") not in ["available_velocity_derived_comparator"]:
        reasons.append(str(row.get("cellrank_status_reason", "CellRank comparator unavailable or unmatched")))
    if not reasons:
        reasons.append("no unresolved limitation for this transition under the prespecified checks; no thresholds tuned")
    return "; ".join(reasons)

evidence["limitations"] = evidence.apply(limitation_reason, axis=1)

out_path = OUTPUT_DIR / "intermediates" / "pancreas_scgeo_representation_dynamics.h5ad"
adata.write_h5ad(out_path, compression="gzip")

write_dataframe(OUTPUT_DIR, "03_transition_representation_alignment", alignment)
write_dataframe(OUTPUT_DIR, "03_negative_control_alignment", controls)
write_dataframe(OUTPUT_DIR, "03_representation_agreement", agreement)
write_dataframe(OUTPUT_DIR, "03_cellrank_transition_comparator", cellrank_compare)
write_dataframe(OUTPUT_DIR, "03_negative_control_summary", control_summary)
write_dataframe(OUTPUT_DIR, "03_state_transition_evidence_table", evidence)
write_alt_text(
    OUTPUT_DIR,
    "03_scgeo_representation_dynamics",
    "State-transition evidence table compares original pancreas annotation-supported cluster transitions with source-state RNA velocity directions across PCA20, PCA30, PCA50, diffusion map, and UMAP diagnostics. Reversed-edge, shuffled-velocity, and rotated-velocity controls remain visible. CellRank is reported as a velocity-derived comparator."
)
write_metadata(OUTPUT_DIR, "03_scgeo_representation_dynamics", CONFIG, {
    "input_h5ad": scvelo_summary["output_h5ad"],
    "output_h5ad": rel_display(out_path, ROOT),
    "output_sha256": sha256_file(out_path),
    "velocity_projection_warnings": velocity_projection_warnings,
    "transition_edges": CONFIG["transition_edges"],
    "representations": CONFIG["representations"],
    "negative_controls": CONFIG["negative_controls"],
    "no_artificial_treatment_control": True,
})
version_record(OUTPUT_DIR, "03_scgeo_representation_dynamics", CONFIG, {"output_h5ad_sha256": sha256_file(out_path)})
evidence
